In [ ]:
// hide-on-readme
// prelude

import { spawnSync } from 'child_process';
import { existsSync, mkdirSync, rmSync, readdirSync } from 'fs';
import { readFileSync, writeFileSync } from 'fs';
import { join } from 'path';

const DEBIAN_RELEASE = process.env.DEBIAN_RELEASE || 'testing';
const DEBOOTSTRAP_VARIANT = 'minbase';
const DEBOOTSTRAP_MIRROR = 'http://deb.debian.org/debian';
const LOCAL_DIST_DIR_SRV = join(process.cwd(), 'dist/server/');
const LOCAL_DIST_DIR_CLIENT = join(process.cwd(), 'dist/client/');
const LOCAL_BUILD_DIR = join(process.cwd(), 'build/base/');


type NotebookCell = {
  cell_type: string;
  source: string[] | string;
  id?: string;
  metadata?: Record<string, unknown>;
};

type NotebookFile = {
  cells: NotebookCell[];
  metadata?: Record<string, unknown>;
  nbformat?: number;
  nbformat_minor?: number;
};

function sourceLines(source: string[] | string): string[] {
  return Array.isArray(source) ? source : source.split('\n');
}

function markdownAnchor(text: string): string {
  return text
    .trim()
    .replace(/[\`*_{}[\]()#+.!?:;,\/\\|"']/g, '')
    .replace(/\s+/g, '-');
}

/**
 * Rebuild the notebook Table of Contents from markdown headings.
 *
 * Usage:
 * - updateNotebookToc();
 * - updateNotebookToc('notebook.ipynb');
 */
function updateNotebookToc(notebookPath = 'notebook.ipynb'): void {
  const raw = readFileSync(notebookPath, 'utf-8');
  const notebook = JSON.parse(raw) as NotebookFile;

  const tocCellIndex = notebook.cells.findIndex((cell) => {
    if (cell.cell_type !== 'markdown') {
      return false;
    }
    return sourceLines(cell.source).some((line) => line.trim() === '## Table of Contents');
  });

  if (tocCellIndex === -1) {
    throw new Error('Could not find a markdown cell titled "## Table of Contents".');
  }

  const headings: Array<{ level: number; text: string }> = [];
  for (let i = 0; i < notebook.cells.length; i += 1) {
    if (i === tocCellIndex) {
      continue;
    }

    const cell = notebook.cells[i];
    if (cell.cell_type !== 'markdown') {
      continue;
    }

    for (const line of sourceLines(cell.source)) {
      const match = line.match(/^(#{2,6})\s+(.+?)\s*$/);
      if (!match) {
        continue;
      }

      const level = match[1].length;
      const text = match[2].trim();
      if (text.toLowerCase() === 'table of contents') {
        continue;
      }

      headings.push({ level, text });
    }
  }

  if (headings.length === 0) {
    throw new Error('No markdown headings found to build the table of contents.');
  }

  const minLevel = Math.min(...headings.map((heading) => heading.level));
  const tocLines = [
    '## Table of Contents',
    '',
    ...headings.map((heading) => {
      const depth = Math.max(heading.level - minLevel, 0);
      const indent = '  '.repeat(depth);
      return `${indent}- [${heading.text}](#${markdownAnchor(heading.text)})`;
    }),
  ];

  notebook.cells[tocCellIndex].source = tocLines.map((line) => `${line}\n`);
  writeFileSync(notebookPath, `${JSON.stringify(notebook, null, 4)}\n`, { encoding: 'utf-8' });
}



In [1]:
// hide-on-readme
// prelude
function runSyncCommand(command: string, args: string[], cwd?: string) {
    const result = spawnSync(command, args, {
        cwd,
        stdio: 'pipe',
        shell: false,
        encoding: 'utf8',
    });
    if (result.error) {
        throw result.error;
    }

    if (result.stdout) {
        console.log(result.stdout);
    }
    if (result.stderr) {
        console.error(result.stderr);
    }

    if (result.status !== 0) {
        throw new Error(`Command failed: ${command} ${args.join(' ')} (exit ${result.status})`);
    }
}

function shellQuote(value: string) {
    if (/^[A-Za-z0-9_\/\.-]+$/.test(value)) {
        return value;
    }
    return `'${value.replace(/'/g, `'\\''`)}'`;
}

function runChrootCommand(chrootPath: string | undefined, command: string, args: string[], cwd?: string) {
    if (!chrootPath) {
        console.log(`Running command without chroot: ${command} ${args.join(' ')}`);
        return runSyncCommand(command, args, cwd);
    }

    checkCommandAvailable('chroot');
    if (!existsSync(chrootPath)) {
        throw new Error(`Chroot path does not exist: ${chrootPath}`);
    }

    const shellPath = join(chrootPath, 'bin/sh');
    if (!existsSync(shellPath)) {
        throw new Error(`Chroot target is missing shell: ${shellPath}`);
    }

    const shellCommand = [command, ...args].map(shellQuote).join(' ');
    console.log(`Running in chroot shell: /bin/sh -c ${shellCommand}`);
    return runSyncCommand('chroot', [chrootPath, '/bin/sh', '-c', shellCommand], cwd);
}

function ensureDir(path: string) {
    if (!existsSync(path)) {
        mkdirSync(path, { recursive: true });
    }
}

function isDirEmpty(path: string): boolean {
    if (!existsSync(path)) {
        return true;
    }
    return readdirSync(path).length === 0;
}

function clearDir(path: string) {
    if (existsSync(path)) {
        rmSync(path, { recursive: true, force: true });
    }
    mkdirSync(path, { recursive: true });
}

function checkCommandAvailable(command: string) {
    const result = spawnSync('command', ['-v', command], {
        stdio: 'ignore',
        shell: true,
    });
    if (result.status !== 0) {
        throw new Error(`Required command not found: ${command}`);
    }
}

4:20 - Cannot find name 'spawnSync'.
40:10 - Cannot find name 'existsSync'.
44:23 - Cannot find name 'join'.
45:10 - Cannot find name 'existsSync'.
55:10 - Cannot find name 'existsSync'.
56:9 - Cannot find name 'mkdirSync'.
61:10 - Cannot find name 'existsSync'.
64:12 - Cannot find name 'readdirSync'.
68:9 - Cannot find name 'existsSync'.
69:9 - Cannot find name 'rmSync'.
71:5 - Cannot find name 'mkdirSync'.
75:20 - Cannot find name 'spawnSync'.


In [ ]:
// hide-on-readme
// prelude

import { existsSync, mkdirSync, writeFileSync, chmodSync } from 'fs';
import { join } from 'path';

/**
 * Install a git pre-commit hook that refreshes README.md before commit.
 *
 * Usage:
 * - addPreCommitHook();
 */
function addPreCommitHook(): void {
  const hookDir = join('.git', 'hooks');
  const hookPath = join(hookDir, 'pre-commit');

  if (!existsSync('.git')) {
    throw new Error('No .git directory found. Run this from the repository root.');
  }

  if (!existsSync(hookDir)) {
    mkdirSync(hookDir, { recursive: true });
  }

  const hookScript = [
    '#!/usr/bin/env bash',
    'set -euo pipefail',
    '',
    'make README.md',
    '',
    'git add README.md',
  ].join('\n');

  writeFileSync(hookPath, `${hookScript}\n`, { encoding: 'utf-8' });
  chmodSync(hookPath, 0o755);
}

// Auto Exec
addPreCommitHook();
updateNotebookToc();

# LMPriestley's Notebook

This is a personal project to collect my personal knowledge and small utilities into an obsidian inspired jupyter notebook.

To view the interactive version of this project, please visit [lmpriestley.com](https://freyground.com). Sources available at [github.com/freylint/gbnotebook](https://www.github.com/freylint/gbnotebook).

## Table of Contents

- [License](#License)
- [Packages](#Packages)
  - [NBECS Interface - Notebook ECS interfaces](#NBECS-Interface---Notebook-ECS-interfaces)
  - [NBECS](#NBECS)
- [Articles](#Articles)
- [Notes](#Notes)
- [Images](#Images)
- [Homelab Setup](#Homelab-Setup)
  - [Hardware](#Hardware)
    - [Network](#Network)
    - [Homelab Server](#Homelab-Server)
    - [Thin Client](#Thin-Client)


## License

This project is licensed under the GNU Affero General Public License v3.0.
See [LICENSE](LICENSE) for the full text.

## Packages

## Homelab Setup

### Hardware
#### Network
Commodity home router into a switch.
#### Homelab Server
4u threadripper machine w/ 2x dedicated GPU and 128GB of RAM.
#### Thin Client
Commodity x86 laptops w/ port replicators.


In [ ]:
// Homelab BaseImage Generation script

function buildBaseImage() {
    checkCommandAvailable('debootstrap');
    ensureDir(LOCAL_DIST_DIR_SRV);

    if (existsSync(LOCAL_BUILD_DIR) && !isDirEmpty(LOCAL_BUILD_DIR)) {
        console.log('Skipping debootstrap because target directory already exists and is non-empty:', LOCAL_BUILD_DIR);
        return;
    }

    clearDir(LOCAL_BUILD_DIR);

    console.log('Syncing Debian min image for test build...');
    console.log('Debian release:', DEBIAN_RELEASE);
    console.log('Debootstrap variant:', DEBOOTSTRAP_VARIANT);
    console.log('Output directory:', LOCAL_BUILD_DIR);
    console.log('Mirror:', DEBOOTSTRAP_MIRROR);

    runSyncCommand('debootstrap', [
        '--variant=' + DEBOOTSTRAP_VARIANT,
        DEBIAN_RELEASE,
        LOCAL_BUILD_DIR,
        DEBOOTSTRAP_MIRROR,
    ]);

    console.log('Debootstrap command completed.');
}
function buildClientImage() {
    // RSYNC base to client directory
    ensureDir(LOCAL_DIST_DIR_CLIENT);
    runSyncCommand('rsync', ['-a', LOCAL_BUILD_DIR, LOCAL_DIST_DIR_CLIENT]);

    runChrootCommand(LOCAL_DIST_DIR_CLIENT, 'echo', ['Hello from chroot']); 
}

function buildServerImage() {
    // RSYNC base to server directory
    ensureDir(LOCAL_DIST_DIR_SRV);
    runSyncCommand('rsync', ['-a', LOCAL_BUILD_DIR, LOCAL_DIST_DIR_SRV]);

    runChrootCommand(LOCAL_DIST_DIR_SRV, 'echo', ['Hello from server chroot']);
}

function buildHLImages() {
    buildBaseImage();
    buildClientImage();
    buildServerImage();


    // TODO Setup OSTREE Repository for managing image versions
}

buildHLImages();



7:5 - Cannot find name 'checkCommandAvailable'.
8:5 - Cannot find name 'ensureDir'.
8:15 - Cannot find name 'LOCAL_DIST_DIR_SRV'.
10:9 - Cannot find name 'existsSync'.
10:20 - Cannot find name 'LOCAL_BUILD_DIR'.
10:41 - Cannot find name 'isDirEmpty'.
10:52 - Cannot find name 'LOCAL_BUILD_DIR'.
11:103 - Cannot find name 'LOCAL_BUILD_DIR'.
15:5 - Cannot find name 'clearDir'.
15:14 - Cannot find name 'LOCAL_BUILD_DIR'.
18:36 - Cannot find name 'DEBIAN_RELEASE'.
19:41 - Cannot find name 'DEBOOTSTRAP_VARIANT'.
20:38 - Cannot find name 'LOCAL_BUILD_DIR'.
21:28 - Cannot find name 'DEBOOTSTRAP_MIRROR'.
23:5 - Cannot find name 'runSyncCommand'.
24:24 - Cannot find name 'DEBOOTSTRAP_VARIANT'.
25:9 - Cannot find name 'DEBIAN_RELEASE'.
26:9 - Cannot find name 'LOCAL_BUILD_DIR'.
27:9 - Cannot find name 'DEBOOTSTRAP_MIRROR'.
34:5 - Cannot find name 'ensureDir'.
34:15 - Cannot find name 'LOCAL_DIST_DIR_CLIENT'.
35:5 - Cannot find name 'runSyncCommand'.
35:36 - Cannot find name 'LOCAL_BUILD_DIR'.
35:5